# EDA — Fórmula 1 | Camada Silver

Notebook de Análise Exploratória de Dados (estrutura inicial).

# Etapa 2 — Análise Exploratória de Dados
## Projeto Formula 1 — Interlagos

Este notebook apresenta a análise exploratória dos dados tratados
na camada Silver do projeto.

O objetivo é avaliar a qualidade dos dados, identificar padrões,
relações, anomalias e gerar hipóteses que apoiem as etapas analíticas
e de modelagem.


## 2. Fontes e datasets utilizados

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb
from pathlib import Path

## 3. Carregamento dos dados

        ↓

Código
Configuração da conexão DuckDB/MinIO

        ↓

Código
Carregamento de TODOS os datasets

        ↓

Código
Validação do carregamento

In [6]:
con = duckdb.connect(
    config={"extension_directory": str(Path("notebooks/.duckdb_extensions"))}
)
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
con.execute("""
    SET s3_endpoint='localhost:9000';
    SET s3_access_key_id='admin';
    SET s3_secret_access_key='minioadmin123';
    SET s3_use_ssl=false;
    SET s3_url_style='path';
""")

calendario = con.sql("""
    SELECT *
    FROM read_parquet(
        's3://f1-data-lake/silver/consolidado/calendario.parquet'
    )
""").df()

resultados = con.sql("""
    SELECT *
    FROM read_parquet(
        's3://f1-data-lake/silver/consolidado/resultados.parquet'
    )
""").df()

voltas = con.sql("""
    SELECT *
    FROM read_parquet(
        's3://f1-data-lake/silver/consolidado/voltas.parquet'
    )
""").df()

pit_stops = con.sql("""
    SELECT *
    FROM read_parquet(
        's3://f1-data-lake/silver/consolidado/pit_stops.parquet'
    )
""").df()

pneus = con.sql("""
    SELECT *
    FROM read_parquet(
        's3://f1-data-lake/silver/consolidado/pneus.parquet'
    )
""").df()

clima = con.sql("""
    SELECT *
    FROM read_parquet(
        's3://f1-data-lake/silver/consolidado/clima.parquet'
    )
""").df()

driver_mapping = con.sql("""
    SELECT *
    FROM read_parquet(
        's3://f1-data-lake/silver/driver_mapping/driver_mapping.parquet'
    )
""").df()

print("Datasets carregados com sucesso.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Datasets carregados com sucesso.


In [7]:
datasets = {
    "calendario": calendario,
    "resultados": resultados,
    "voltas": voltas,
    "pit_stops": pit_stops,
    "pneus": pneus,
    "clima": clima,
    "driver_mapping": driver_mapping
}

for nome, df in datasets.items():
    print(f"{nome:<15} {len(df):>8,} registros | {len(df.columns):>2} colunas")

calendario           233 registros | 19 colunas
resultados           202 registros | 36 colunas
voltas            12,589 registros | 20 colunas
pit_stops            512 registros | 20 colunas
pneus              8,815 registros | 15 colunas
clima              1,117 registros | 17 colunas
driver_mapping       137 registros |  9 colunas


## 4. Validação da Qualidade dos Dados



### 4.1 Visão geral dos datasets

Inicialmente, são avaliados o volume, a quantidade de atributos e o
consumo aproximado de memória de cada dataset carregado da camada Silver.

Essa visão permite compreender a dimensão das bases utilizadas e suas
diferentes granularidades antes das análises exploratórias.

In [10]:
resumo = []

for nome, df in datasets.items():
    resumo.append({
        "dataset": nome,
        "registros": len(df),
        "colunas": len(df.columns),
        "memoria_mb": df.memory_usage(deep=True).sum() / 1024**2
    })

resumo_datasets = (
    pd.DataFrame(resumo)
      .sort_values("registros", ascending=False)
      .reset_index(drop=True)
)

resumo_datasets["memoria_mb"] = resumo_datasets["memoria_mb"].round(2)

display(resumo_datasets)

,dataset,registros,colunas,memoria_mb
0,voltas,12589,20,8.36
1,pneus,8815,15,4.46
2,clima,1117,17,0.51
3,pit_stops,512,20,0.38
4,calendario,233,19,0.16
5,resultados,202,36,0.24
6,driver_mapping,137,9,0.06


In [11]:
granularidades = {
    "calendario": "1 registro por corrida",
    "resultados": "1 piloto por corrida",
    "voltas": "1 piloto por volta",
    "pit_stops": "1 pit stop por piloto",
    "pneus": "1 piloto por volta/sessão",
    "clima": "1 medição meteorológica por instante",
    "driver_mapping": "1 piloto FastF1 por temporada"
}

resumo_datasets["granularidade"] = (
    resumo_datasets["dataset"].map(granularidades)
)

display(resumo_datasets)

,dataset,registros,colunas,memoria_mb,granularidade
0,voltas,12589,20,8.36,1 piloto por volta
1,pneus,8815,15,4.46,1 piloto por volta/sessão
2,clima,1117,17,0.51,1 medição meteorológica por instante
3,pit_stops,512,20,0.38,1 pit stop por piloto
4,calendario,233,19,0.16,1 registro por corrida
5,resultados,202,36,0.24,1 piloto por corrida
6,driver_mapping,137,9,0.06,1 piloto FastF1 por temporada


### 4.2 Valores ausentes

A presença de valores ausentes foi avaliada em todos os datasets.

Valores nulos não são necessariamente problemas de qualidade. Alguns
podem ser esperados devido às características da Fórmula 1 ou à
indisponibilidade de determinados atributos na fonte.

Por isso, além da quantidade de nulos, será analisado o contexto das
variáveis afetadas.

In [15]:
resumo_nulos = []

for nome, df in datasets.items():
    for coluna in df.columns:

        qtd_nulos = df[coluna].isna().sum()

        if qtd_nulos > 0:
            resumo_nulos.append({
                "dataset": nome,
                "coluna": coluna,
                "nulos": qtd_nulos,
                "percentual": round(qtd_nulos / len(df) * 100, 2)
            })

resumo_nulos = (
    pd.DataFrame(resumo_nulos)
      .sort_values(
          ["dataset", "percentual"],
          ascending=[True, False]
      )
)

display(resumo_nulos)

,dataset,coluna,nulos,percentual
1,calendario,third_practice_date,24,10.30
0,calendario,second_practice_date,18,7.73
2,resultados,race_time,81,40.10
3,resultados,race_time_millis,81,40.10
8,resultados,fastest_lap_average_speed,33,16.34
9,resultados,fastest_lap_average_speed_unit,33,16.34
4,resultados,fastest_lap_rank,14,6.93
5,resultados,fastest_lap,14,6.93
6,resultados,fastest_lap_time,14,6.93
7,resultados,fastest_lap_time_millis,14,6.93


4.2.1 - Analise de nulos presente em racetime

In [16]:
race_time_nulos = (
    resultados[resultados["race_time"].isna()]
    .groupby("status")
    .size()
    .reset_index(name="quantidade")
    .sort_values("quantidade", ascending=False)
)

display(race_time_nulos)

,status,quantidade
0,+1 Lap,34
14,Retired,12
1,+2 Laps,8
4,Collision,8
3,Accident,3
6,Did not start,3
2,+4 Laps,2
5,Collision damage,2
7,Disqualified,1
8,Engine,1


In [17]:
display(
    pd.crosstab(
        resultados["status"],
        resultados["race_time"].isna(),
        margins=True
    )
)

race_time,False,True,All
status,,,
+1 Lap,0,34,34
+2 Laps,0,8,8
+4 Laps,0,2,2
Accident,0,3,3
Collision,0,8,8
Collision damage,0,2,2
Did not start,0,3,3
Disqualified,0,1,1
Engine,0,1,1


4.2.2 - Analise de nulos presente em fastest_lap

In [18]:
cols_fastest_lap = [
    "fastest_lap_rank",
    "fastest_lap",
    "fastest_lap_time",
    "fastest_lap_time_millis",
    "fastest_lap_average_speed",
    "fastest_lap_average_speed_unit"
]

resultados[cols_fastest_lap].isna().sum().to_frame("nulos")

,nulos
fastest_lap_rank,14
fastest_lap,14
fastest_lap_time,14
fastest_lap_time_millis,14
fastest_lap_average_speed,33
fastest_lap_average_speed_unit,33


In [19]:
velocidade_ausente = resultados[
    resultados["fastest_lap"].notna()
    & resultados["fastest_lap_average_speed"].isna()
]

display(
    velocidade_ausente[
        [
            "season",
            "driver_id",
            "status",
            "fastest_lap",
            "fastest_lap_time",
            "fastest_lap_average_speed"
        ]
    ]
)

,season,driver_id,status,fastest_lap,fastest_lap_time,fastest_lap_average_speed
182,2025,hulkenberg,Finished,39,1:13.474,NaN
183,2025,alonso,Finished,48,1:13.312,NaN
184,2025,albon,Finished,59,1:12.400,NaN
186,2025,russell,Finished,52,1:13.097,NaN
187,2025,gasly,Finished,42,1:13.736,NaN
188,2025,colapinto,Finished,45,1:12.816,NaN
189,2025,ocon,Finished,10,1:13.481,NaN
190,2025,leclerc,Retired,2,1:43.560,NaN
191,2025,antonelli,Finished,52,1:13.123,NaN
192,2025,max_verstappen,Finished,56,1:12.447,NaN


In [20]:
display(
    velocidade_ausente
    .groupby("season")
    .size()
    .reset_index(name="quantidade")
)

,season,quantidade
0,2025,19


4.2.3 - Analise de nulos presente em second_practice e third_practice

In [22]:
treinos_nulos = (
    calendario
    .groupby("season")
    .agg(
        corridas=("round", "count"),
        second_practice_nulos=("second_practice_date", lambda x: x.isna().sum()),
        third_practice_nulos=("third_practice_date", lambda x: x.isna().sum())
    )
    .reset_index()
)

display(treinos_nulos)

,season,corridas,second_practice_nulos,third_practice_nulos
0,2015,19,0,0
1,2016,21,0,0
2,2017,20,0,0
3,2018,21,0,0
4,2019,21,0,0
5,2020,17,0,0
6,2021,22,0,3
7,2022,22,0,3
8,2023,22,6,6
9,2024,24,6,6


#### Conclusão sobre valores ausentes

A análise demonstrou que os valores ausentes identificados não representam,
no contexto atual, problemas críticos de qualidade dos dados.

- **Datas de treinos livres:** os valores ausentes são esperados e estão
  associados a eventos cujo formato não contou com múltiplas sessões de
  treino livre.

- **Tempo final de corrida:** os valores ausentes estão associados aos
  pilotos que não completaram a corrida. Portanto, a ausência do tempo final
  representa uma característica do resultado esportivo, e não uma falha
  de preenchimento.

- **Informações de fastest lap:** as ausências identificadas estão
  concentradas no retorno de 2025, caracterizando uma limitação de
  disponibilidade dos dados fornecidos pela API Jolpica.

Dessa forma, não será realizada imputação dos valores ausentes. Os campos
serão mantidos como nulos, preservando o significado original das fontes.

### 4.3 Registros duplicados

A análise de duplicidade foi realizada considerando as chaves lógicas
e a granularidade esperada de cada dataset.

Essa abordagem permite identificar registros que representam a mesma
entidade ou evento, mesmo quando outros atributos apresentam valores
diferentes.

In [24]:
chaves_logicas = {
    "calendario": ["season", "round"],
    "resultados": ["season", "round", "driver_id"],
    "voltas": ["season", "round", "driver_id", "lap"],
    "pit_stops": ["season", "round", "driver_id", "stop"],
    "driver_mapping": ["season", "fastf1_driver_id"]
}

resultado_duplicidades = []

for nome, chave in chaves_logicas.items():
    df = datasets[nome]

    grupos_duplicados = (
        df.groupby(chave, dropna=False)
          .size()
          .reset_index(name="quantidade")
          .query("quantidade > 1")
    )

    resultado_duplicidades.append({
        "dataset": nome,
        "chave_logica": " + ".join(chave),
        "grupos_duplicados": len(grupos_duplicados),
        "registros_envolvidos": grupos_duplicados["quantidade"].sum()
            if not grupos_duplicados.empty else 0
    })

duplicidades = pd.DataFrame(resultado_duplicidades)

display(duplicidades)

,dataset,chave_logica,grupos_duplicados,registros_envolvidos
0,calendario,season + round,0,0
1,resultados,season + round + driver_id,0,0
2,voltas,season + round + driver_id + lap,0,0
3,pit_stops,season + round + driver_id + stop,0,0
4,driver_mapping,season + fastf1_driver_id,0,0


In [25]:
chave_pneus = [
    "season",
    "session",
    "driver_id",
    "lap_number"
]

duplicados_pneus = (
    pneus
    .groupby(chave_pneus, dropna=False)
    .size()
    .reset_index(name="quantidade")
    .query("quantidade > 1")
)

print(f"Grupos duplicados: {len(duplicados_pneus)}")
print(
    f"Registros envolvidos: "
    f"{duplicados_pneus['quantidade'].sum() if not duplicados_pneus.empty else 0}"
)

display(duplicados_pneus)

Grupos duplicados: 0
Registros envolvidos: 0


,season,session,driver_id,lap_number,quantidade


In [26]:
chave_pneus = [
    "season",
    "session",
    "driver_id",
    "lap_number"
]

duplicados_pneus = (
    pneus
    .groupby(chave_pneus, dropna=False)
    .size()
    .reset_index(name="quantidade")
    .query("quantidade > 1")
)

print(f"Grupos duplicados: {len(duplicados_pneus)}")
print(
    f"Registros envolvidos: "
    f"{duplicados_pneus['quantidade'].sum() if not duplicados_pneus.empty else 0}"
)

display(duplicados_pneus)

Grupos duplicados: 0
Registros envolvidos: 0


,season,session,driver_id,lap_number,quantidade


#### Conclusão sobre registros duplicados

A análise de duplicidade foi realizada considerando as chaves lógicas e
a granularidade esperada de cada dataset.

Não foram identificados registros duplicados em nenhuma das bases
avaliadas:

- calendário;
- resultados;
- voltas;
- pit stops;
- pneus;
- clima;
- driver mapping.

O resultado indica que as chaves lógicas definidas para os datasets são
respeitadas e que não há evidências de duplicidades capazes de provocar
contagens incorretas ou relacionamentos muitos-para-muitos indesejados
nas análises subsequentes.

Dessa forma, não foi necessário aplicar tratamento adicional de
deduplicação na camada Silver.

### 4.4 Integridade entre os datasets

Após a validação das chaves lógicas, foi analisada a integridade dos
relacionamentos entre os datasets da camada Silver.

O objetivo é identificar registros órfãos, isto é, fatos que não possuem
correspondência nas entidades utilizadas como referência.

Foram avaliados os principais relacionamentos necessários para as
análises posteriores.

In [40]:
integridade = []

# ------------------------------------------------------------
# Resultados -> Calendário
# ------------------------------------------------------------

resultado_calendario = resultados.merge(
    calendario[["season", "round"]].drop_duplicates(),
    on=["season", "round"],
    how="left",
    indicator=True
)

orf_resultados = (
    resultado_calendario["_merge"] == "left_only"
).sum()

integridade.append({
    "relacionamento": "resultados → calendario",
    "registros_origem": len(resultados),
    "orfãos": orf_resultados,
    "cobertura_pct": round(
        (1 - orf_resultados / len(resultados)) * 100, 2
    )
})


# ------------------------------------------------------------
# Voltas -> Resultados
# ------------------------------------------------------------

volta_resultado = voltas.merge(
    resultados[
        ["season", "round", "driver_id"]
    ].drop_duplicates(),
    on=["season", "round", "driver_id"],
    how="left",
    indicator=True
)

orf_voltas = (
    volta_resultado["_merge"] == "left_only"
).sum()

integridade.append({
    "relacionamento": "voltas → resultados",
    "registros_origem": len(voltas),
    "orfãos": orf_voltas,
    "cobertura_pct": round(
        (1 - orf_voltas / len(voltas)) * 100, 2
    )
})


# ------------------------------------------------------------
# Pit Stops -> Resultados
# ------------------------------------------------------------

pit_resultado = pit_stops.merge(
    resultados[
        ["season", "round", "driver_id"]
    ].drop_duplicates(),
    on=["season", "round", "driver_id"],
    how="left",
    indicator=True
)

orf_pit = (
    pit_resultado["_merge"] == "left_only"
).sum()

integridade.append({
    "relacionamento": "pit_stops → resultados",
    "registros_origem": len(pit_stops),
    "orfãos": orf_pit,
    "cobertura_pct": round(
        (1 - orf_pit / len(pit_stops)) * 100, 2
    )
})


# ------------------------------------------------------------
# Pneus -> Driver Mapping
# ------------------------------------------------------------

pneu_mapping = pneus.merge(
    driver_mapping[
        ["season", "fastf1_driver_id"]
    ].drop_duplicates(),
    left_on=["season", "driver_id"],
    right_on=["season", "fastf1_driver_id"],
    how="left",
    indicator=True
)

orf_pneus = (
    pneu_mapping["_merge"] == "left_only"
).sum()

integridade.append({
    "relacionamento": "pneus → driver_mapping",
    "registros_origem": len(pneus),
    "orfãos": orf_pneus,
    "cobertura_pct": round(
        (1 - orf_pneus / len(pneus)) * 100, 2
    )
})


# Resultado
integridade_df = pd.DataFrame(integridade)

display(integridade_df)

,relacionamento,registros_origem,orfãos,cobertura_pct
0,resultados → calendario,202,0,100.0
1,voltas → resultados,12589,0,100.0
2,pit_stops → resultados,512,0,100.0
3,pneus → driver_mapping,8815,0,100.0


#### Conclusão sobre integridade entre os datasets

Os principais relacionamentos entre os datasets da camada Silver foram
validados com o objetivo de identificar possíveis registros órfãos.

Foram avaliados os seguintes relacionamentos:

- `resultados → calendario`
- `voltas → resultados`
- `pit_stops → resultados`
- `pneus → driver_mapping`

Todos os relacionamentos apresentaram cobertura de **100%**, não sendo
identificados registros órfãos nas chaves avaliadas.

Esse resultado indica consistência referencial entre os principais
datasets utilizados nas análises posteriores.

A associação entre `pneus` e `voltas` é tratada separadamente, pois envolve
fontes distintas e possui particularidades relacionadas aos registros de
abandono.

#### 4.4.1 Cobertura entre pneus e voltas

Os datasets de pneus e voltas são provenientes de fontes distintas e
utilizam identificadores diferentes para os pilotos.

Após a utilização da dimensão `driver_mapping`, 100% dos 8.815 registros
elegíveis de pneus foram associados a um piloto da base Jolpica.

Ao acrescentar o número da volta ao relacionamento:

- Registros de pneus: **8.815**
- Registros com volta correspondente: **8.803**
- Registros sem volta correspondente: **12**
- Cobertura: **99,86%**

Os 12 registros sem correspondência foram analisados individualmente e
estão associados a abandonos.

Nesses casos, a FastF1 mantém informação do pneu utilizado na volta do
incidente, enquanto a Jolpica registra apenas as voltas efetivamente
concluídas pelo piloto.

Portanto, os registros sem correspondência não foram classificados como
falhas de integração.

Para análises que necessitem simultaneamente de informações de pneus e
voltas, será utilizado `INNER JOIN`, mantendo 99,86% da cobertura.

IDs incompatíveis

      ↓

driver_mapping

      ↓

137/137 pilotos mapeados

      ↓

join por volta

      ↓

8.803/8.815 = 99,86%

      ↓

12 diferenças explicadas por abandonos

#### 4.4.2 Compatibilidade de identificadores entre FastF1 e Jolpica

Os datasets de pneus e voltas utilizam identificadores diferentes para os pilotos.

A FastF1 utiliza códigos curtos, como `VER`, `HAM` e `LEC`, enquanto a Jolpica utiliza identificadores canônicos, como `max_verstappen`, `hamilton` e `leclerc`.

Por esse motivo, o relacionamento direto entre `pneus.driver_id` e `voltas.driver_id` não é válido.

Foi criada a dimensão `driver_mapping` para compatibilizar os identificadores entre as duas fontes.

In [41]:
comparacao_ids = (
    pneus[["season", "driver_id"]]
    .drop_duplicates()
    .rename(columns={"driver_id": "fastf1_driver_id"})
    .merge(
        resultados[["season", "driver_id"]]
        .drop_duplicates()
        .rename(columns={"driver_id": "jolpica_driver_id"}),
        on="season",
        how="inner"
    )
)

comparacao_ids["match_direto"] = (
    comparacao_ids["fastf1_driver_id"]
    == comparacao_ids["jolpica_driver_id"]
)

display(
    comparacao_ids[
        ["season", "fastf1_driver_id", "jolpica_driver_id", "match_direto"]
    ].head(20)
)

,season,fastf1_driver_id,jolpica_driver_id,match_direto
0,2018,HAM,perez,False
1,2018,HAM,stroll,False
2,2018,HAM,grosjean,False
3,2018,HAM,ericsson,False
4,2018,HAM,hamilton,False
5,2018,HAM,kevin_magnussen,False
6,2018,HAM,gasly,False
7,2018,HAM,vettel,False
8,2018,HAM,max_verstappen,False
9,2018,HAM,leclerc,False


In [42]:
ids_fastf1 = (
    pneus[["season", "driver_id"]]
    .drop_duplicates()
    .rename(columns={"driver_id": "fastf1_driver_id"})
)

ids_jolpica = (
    resultados[["season", "driver_id"]]
    .drop_duplicates()
    .rename(columns={"driver_id": "jolpica_driver_id"})
)

print("Exemplos FastF1:")
display(ids_fastf1.head(10))

print("Exemplos Jolpica:")
display(ids_jolpica.head(10))

Exemplos FastF1:


,season,fastf1_driver_id
0,2018,HAM
71,2018,VER
142,2018,RAI
213,2018,RIC
284,2018,BOT
355,2018,VET
426,2018,LEC
497,2018,GRO
568,2018,MAG
639,2018,PER


Exemplos Jolpica:


,season,jolpica_driver_id
0,2015,rosberg
1,2015,perez
2,2015,stevens
3,2015,sainz
4,2015,bottas
5,2015,maldonado
6,2015,nasr
7,2015,hamilton
8,2015,button
9,2015,ricciardo


In [43]:
match_direto = ids_fastf1.merge(
    ids_jolpica,
    left_on=["season", "fastf1_driver_id"],
    right_on=["season", "jolpica_driver_id"],
    how="left",
    indicator=True
)

resumo_match_direto = (
    match_direto["_merge"]
    .value_counts()
    .rename_axis("resultado")
    .reset_index(name="quantidade")
)

display(resumo_match_direto)

,resultado,quantidade
0,left_only,137
1,right_only,0
2,both,0


In [44]:
validacao_mapping = (
    ids_fastf1
    .merge(
        driver_mapping[
            [
                "season",
                "fastf1_driver_id",
                "jolpica_driver_id",
                "match_status"
            ]
        ],
        on=["season", "fastf1_driver_id"],
        how="left"
    )
)

display(validacao_mapping.head(15))

,season,fastf1_driver_id,jolpica_driver_id,match_status
0,2018,HAM,hamilton,MATCH_OK
1,2018,VER,max_verstappen,MATCH_OK
2,2018,RAI,raikkonen,MATCH_OK
3,2018,RIC,ricciardo,MATCH_OK
4,2018,BOT,bottas,MATCH_OK
5,2018,VET,vettel,MATCH_OK
6,2018,LEC,leclerc,MATCH_OK
7,2018,GRO,grosjean,MATCH_OK
8,2018,MAG,kevin_magnussen,MATCH_OK
9,2018,PER,perez,MATCH_OK


In [45]:
resumo_mapping = (
    validacao_mapping["match_status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="quantidade")
)

display(resumo_mapping)

,status,quantidade
0,MATCH_OK,137


#### Conclusão sobre compatibilidade dos identificadores

A análise confirmou que os identificadores de pilotos utilizados pela FastF1 e pela Jolpica não são diretamente compatíveis.

A FastF1 utiliza códigos curtos de três caracteres, enquanto a Jolpica utiliza identificadores textuais próprios.

Por esse motivo, um relacionamento direto entre `pneus.driver_id` e `voltas.driver_id` não seria válido.

A dimensão `driver_mapping` foi utilizada como camada de compatibilização entre as fontes.

No recorte atual, todos os 137 pares piloto-temporada presentes nos dados válidos da FastF1 foram associados com sucesso a um identificador Jolpica, sem casos ambíguos ou sem correspondência.

### 4.5 Cobertura temporal

Os datasets utilizados possuem diferentes períodos de disponibilidade,
devido às características e limitações das fontes de origem.

Nesta etapa é avaliada a cobertura histórica de cada dataset, permitindo
definir quais temporadas podem ser utilizadas em cada tipo de análise.

A ausência de uma temporada não será interpretada automaticamente como
problema de qualidade, pois pode representar indisponibilidade da fonte
ou inexistência do evento analisado.

In [46]:
cobertura_temporal = []

for nome, df in datasets.items():

    temporadas = sorted(df["season"].dropna().unique())

    cobertura_temporal.append({
        "dataset": nome,
        "temporada_inicial": min(temporadas),
        "temporada_final": max(temporadas),
        "qtd_temporadas": len(temporadas),
        "temporadas_disponiveis": ", ".join(map(str, temporadas))
    })

cobertura_temporal_df = pd.DataFrame(cobertura_temporal)

display(cobertura_temporal_df)

,dataset,temporada_inicial,temporada_final,qtd_temporadas,temporadas_disponiveis
0,calendario,2015,2025,11,"2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022..."
1,resultados,2015,2025,10,"2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023..."
2,voltas,2015,2025,10,"2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023..."
3,pit_stops,2015,2025,10,"2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023..."
4,pneus,2018,2025,7,"2018, 2019, 2021, 2022, 2023, 2024, 2025"
5,clima,2018,2025,7,"2018, 2019, 2021, 2022, 2023, 2024, 2025"
6,driver_mapping,2018,2025,7,"2018, 2019, 2021, 2022, 2023, 2024, 2025"


In [47]:
temporadas = sorted(
    set().union(*[
        set(df["season"].dropna().unique())
        for df in datasets.values()
    ])
)

matriz_cobertura = pd.DataFrame(
    index=datasets.keys(),
    columns=temporadas
)

for nome, df in datasets.items():

    temporadas_dataset = set(df["season"].dropna().unique())

    for season in temporadas:
        matriz_cobertura.loc[nome, season] = (
            "✓" if season in temporadas_dataset else "—"
        )

display(matriz_cobertura)

,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
calendario,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓,✓
resultados,✓,✓,✓,✓,✓,—,✓,✓,✓,✓,✓
voltas,✓,✓,✓,✓,✓,—,✓,✓,✓,✓,✓
pit_stops,✓,✓,✓,✓,✓,—,✓,✓,✓,✓,✓
pneus,—,—,—,✓,✓,—,✓,✓,✓,✓,✓
clima,—,—,—,✓,✓,—,✓,✓,✓,✓,✓
driver_mapping,—,—,—,✓,✓,—,✓,✓,✓,✓,✓


#### Conclusão sobre cobertura temporal

A análise da cobertura temporal demonstrou que os datasets possuem
diferentes períodos de disponibilidade.

O dataset `calendario` possui cobertura completa entre 2015 e 2025,
incluindo 2020. Entretanto, os datasets relacionados aos dados efetivos
das corridas (`resultados`, `voltas` e `pit_stops`) não possuem registros
para 2020, uma vez que não houve Grande Prêmio de Fórmula 1 em Interlagos
naquela temporada.

Os dados provenientes da FastF1 (`pneus` e `clima`) apresentam cobertura
a partir de 2018, também sem registros para 2020. A dimensão
`driver_mapping`, criada para compatibilizar os identificadores entre
FastF1 e Jolpica, acompanha o mesmo período de disponibilidade.

Dessa forma, o período utilizado nas análises posteriores deverá ser
definido de acordo com os datasets necessários:

- análises baseadas em resultados, voltas e pit stops poderão utilizar
  2015–2019 e 2021–2025;
- análises que dependam de pneus ou clima utilizarão
  2018–2019 e 2021–2025.

As diferenças de cobertura serão consideradas na interpretação dos
resultados para evitar comparações entre períodos que não possuem
disponibilidade equivalente de dados.

### 4.6 Problemas identificados e tratamentos realizados

Durante a ingestão, transformação e análise de qualidade foram
identificadas diferenças de estrutura, cobertura e granularidade entre
as fontes utilizadas.

Os principais problemas encontrados e as decisões adotadas são
consolidados a seguir, permitindo documentar as limitações conhecidas
antes do início das análises exploratórias.

In [48]:
problemas_qualidade = pd.DataFrame([
    {
        "problema": "Paginação dos pit stops",
        "impacto": "Quantidade de registros inferior ao esperado",
        "tratamento": "Implementação de paginação na API Jolpica",
        "resultado": "512 registros e validação OK"
    },
    {
        "problema": "Identificadores de pilotos incompatíveis",
        "impacto": "FastF1 e Jolpica não permitem relacionamento direto",
        "tratamento": "Criação da dimensão driver_mapping",
        "resultado": "137/137 pares mapeados"
    },
    {
        "problema": "Registro FastF1 inválido para 2020",
        "impacto": "GP da Turquia identificado incorretamente como Interlagos",
        "tratamento": "Validação do evento e remoção dos registros inválidos",
        "resultado": "2020 removido de pneus e clima"
    },
    {
        "problema": "Cobertura pneus × voltas",
        "impacto": "12 registros de pneus sem volta correspondente",
        "tratamento": "Investigação dos registros sem correspondência",
        "resultado": "8.803/8.815 registros relacionados (99,86%)"
    },
    {
        "problema": "Alinhamento temporal clima × voltas",
        "impacto": "Fontes não possuem chave temporal diretamente compatível",
        "tratamento": "Relacionamento temporal estimado apenas na camada analítica",
        "resultado": "Não persistido na Silver"
    },
    {
        "problema": "Valores ausentes",
        "impacto": "Campos específicos apresentam valores nulos",
        "tratamento": "Análise contextual sem imputação automática",
        "resultado": "Ausências documentadas e preservadas"
    },
    {
        "problema": "Registros duplicados",
        "impacto": "Possível alteração de métricas e relacionamentos",
        "tratamento": "Validação pelas chaves lógicas",
        "resultado": "Nenhuma duplicidade identificada"
    },
    {
        "problema": "Integridade referencial",
        "impacto": "Possibilidade de registros órfãos",
        "tratamento": "Validação dos principais relacionamentos",
        "resultado": "100% de cobertura"
    }
])

display(problemas_qualidade)

,problema,impacto,tratamento,resultado
0,Paginação dos pit stops,Quantidade de registros inferior ao esperado,Implementação de paginação na API Jolpica,512 registros e validação OK
1,Identificadores de pilotos incompatíveis,FastF1 e Jolpica não permitem relacionamento d...,Criação da dimensão driver_mapping,137/137 pares mapeados
2,Registro FastF1 inválido para 2020,GP da Turquia identificado incorretamente como...,Validação do evento e remoção dos registros in...,2020 removido de pneus e clima
3,Cobertura pneus × voltas,12 registros de pneus sem volta correspondente,Investigação dos registros sem correspondência,"8.803/8.815 registros relacionados (99,86%)"
4,Alinhamento temporal clima × voltas,Fontes não possuem chave temporal diretamente ...,Relacionamento temporal estimado apenas na cam...,Não persistido na Silver
5,Valores ausentes,Campos específicos apresentam valores nulos,Análise contextual sem imputação automática,Ausências documentadas e preservadas
6,Registros duplicados,Possível alteração de métricas e relacionamentos,Validação pelas chaves lógicas,Nenhuma duplicidade identificada
7,Integridade referencial,Possibilidade de registros órfãos,Validação dos principais relacionamentos,100% de cobertura


#### Conclusão da análise de qualidade

As validações realizadas demonstraram que os datasets da camada Silver
apresentam qualidade adequada para o desenvolvimento da análise
exploratória.

Os principais problemas encontrados durante a preparação dos dados foram
investigados e tratados ou, quando decorrentes das características das
fontes, documentados como limitações conhecidas.

Não foram identificadas duplicidades nas chaves lógicas avaliadas e os
principais relacionamentos apresentaram 100% de integridade referencial.

As diferenças de cobertura entre FastF1 e Jolpica foram documentadas,
assim como as particularidades do relacionamento entre pneus e voltas.

O relacionamento temporal entre clima e voltas permanece como uma
limitação conhecida. Como as fontes não possuem uma chave temporal
diretamente compatível, qualquer associação será tratada como estimativa
na camada analítica, sem alteração dos dados originais da camada Silver.

Com essas considerações, os dados são considerados adequados para o
prosseguimento da análise exploratória, respeitando as limitações
documentadas.

## 5. Tratamentos e limitações conhecidas

## 6. Análise descritiva

## 7. Análise de resultados e classificação

## 8. Análise de voltas

## 9. Análise de pit stops

## 10. Análise de pneus

## 11. Análise de clima

## 12. Análises combinadas

## 13. Correlações e padrões

## 14. Anomalias e outliers

## 15. Insights de negócio

## 16. Hipóteses para modelagem

## 17. Limitações

## 18. Conclusão